#Final Report Written by AI

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import sys
sys.path.append('../')

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression as SklearnLR
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.decomposition import PCA as SklearnPCA

from minilearn.classifiers import LogisticRegression as MiniLR
from minilearn.classifiers import GaussianNaiveBayes as MiniNB
from minilearn.classifiers import KNearestNeighbors as MiniKNN
from minilearn.classifiers import SVM as MiniSVM
from minilearn.classifiers import DecisionTree as MiniDT
from minilearn.classifiers import KMeans as MiniKMeans
from minilearn.decomposition import PCA as MiniPCA
from minilearn.neural_network import ANN as MiniANN
from minilearn.metrics import (accuracy_score as mini_accuracy, precision_score as mini_precision, recall_score as mini_recall, f1_score as mini_f1, classification_report as mini_report)



# Speech Emotion Recognition (SER) Using the RAVDESS Dataset
## CSE 432/532 — Machine Learning Semester Project

---

## Introduction

### What is Speech Emotion Recognition?
Speech Emotion Recognition (SER) is the task of automatically identifying 
the emotional state of a speaker from their voice. This is a challenging 
problem in machine learning because emotions are complex, subjective, and 
often expressed subtly through variations in pitch, energy, rhythm, and 
spectral characteristics of speech.

SER has many real world applications including:
- Mental health monitoring and therapy assistance
- Customer service quality assessment
- Human-computer interaction
- Driver safety monitoring
- Sentiment analysis in call centers

### The RAVDESS Dataset
This project uses the Ryerson Audio-Visual Database of Emotional Speech 
and Song (RAVDESS), a validated multimodal database of emotional speech 
and song recordings. We use the audio-only portion of the dataset.

**Dataset Statistics:**
- 24 professional actors (12 male, 12 female)
- 2 vocal channels: speech and song
- 8 emotional categories: neutral, calm, happy, sad, angry, fearful, 
disgust, and surprised
- 2,452 total audio files (1,440 speech + 1,012 song)
- 16-bit, 48kHz WAV format

### Project Overview
This project builds an end-to-end SER system that:
1. Extracts meaningful audio features from raw recordings
2. Implements core ML algorithms from scratch in a custom library called MiniLearn
3. Applies and compares classical and modern classification techniques
4. Evaluates models using rigorous cross-validation and metrics
5. Explores unsupervised learning and dimensionality reduction

### MiniLearn Library
A key component of this project is MiniLearn — a custom Python ML library 
built from scratch. MiniLearn implements the following algorithms:
- **Preprocessing:** StandardScaler, train_test_split, KFold
- **Classifiers:** Logistic Regression, Gaussian Naive Bayes, KNN, SVM, Decision Tree, KMeans, ANN
- **Regression:** Linear Regression
- **Decomposition:** PCA
- **Metrics:** accuracy, precision, recall, F1 score, confusion matrix, classification report

In [ ]:

## 2. Feature Extraction

### 2.1 Hand-Crafted Audio Features
For each audio file, the following features were extracted using the librosa 
library. Since each audio file has a different length, summary statistics 
(mean and standard deviation) were computed over frames to produce a 
fixed-length feature vector.

| Feature | Description | # Values |
|---|---|---|
| MFCCs (13 coefficients) | Compact spectral envelope representation | 26 |
| MFCC Deltas (1st order) | Rate of change of MFCCs | 26 |
| MFCC Delta-Deltas (2nd order) | Acceleration of MFCCs | 26 |
| Zero Crossing Rate | Rate of signal sign changes | 2 |
| RMS Energy | Loudness per frame | 2 |
| Spectral Centroid | Center of mass of spectrum | 2 |
| Spectral Bandwidth | Width of spectral band | 2 |
| Spectral Rolloff | Frequency of 85% energy | 2 |
| Chroma Features | 12 pitch class profile | 24 |
| **Total** | | **112** |

### 2.2 Feature Standardization
All features were standardized using z-score normalization before classification. 
To avoid data leakage, the scaler was fit on training data only and then applied 
to both training and test sets.

### 2.3 Train/Test Split
The dataset was split into 80% training (1,961 samples) and 20% test (491 samples) 
using stratified splitting to maintain class proportions.